In [ ]:
import networkx as nx
import osmnx as ox
import geopandas as gpd
import contextily as ctx 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Traveling Salesperson Problem
The canonical [Traveling Salesperson Problem](https://en.wikipedia.org/wiki/Travelling_salesman_problem) is stated as:
>  "Given a list of cities and the distances between each pair of cities, what is the shortest possible route that visits each city and returns to the origin city?"

This is generalizable to finding the shortest [Hamiltonian cycle](http://mathworld.wolfram.com/HamiltonianCycle.html) on a fully connected graph (i.e. all nodes can be reached from all other nodes).

This problem is [NP-hard](https://en.wikipedia.org/wiki/P_versus_NP_problem), meaning it is not possible for an algorithm to solve all instances of the problem quickly (i.e. in polynomial time). However, there are many approximate and heuristic approaches which can give reasonable solutions in shorter time.

In [ ]:
place_name = 'New York City, NY, United States'
# place_roads = ox.graph_from_place(place_name)

I've already downloaded the roads for NYC and it's hosted on our file server. However, if you wanted to do it for yourself, you can uncomment the next cell. It will take a long time to run, however, since it is a big set of roads.

In [ ]:
# # We can query osm for the roads
# place_roads = ox.graph_from_place(place_name)
# # save graph to file for reuse
# ox.io.save_graphml(place_roads, 'nyc_osmnx.graphml')

In [ ]:
!wget https://files.bwsi-remote-sensing.net/data/nyc_osmnx.graphml

In [ ]:
# loading graph from a file. This will take a while to load.
place_roads = ox.io.load_graphml('nyc_osmnx.graphml')

In [ ]:
place_roads_nodes, place_roads_edges = ox.graph_to_gdfs(place_roads)

In [ ]:
place_roads_nodes

In [ ]:
fig = plt.figure(figsize=[10,10])
ax = fig.add_subplot(1,1,1)
place_roads_edges.plot(ax=ax, color=[0, 0, 0], linewidth=0.5)

Let's say you wanted to do a ice cream crawl: you want to visit every ice cream shop in a city. What is the shortest route that you would take that takes you to every ice cream shop in a city and brings you back to your starting point?

In [ ]:
# Get ice cream shop locations
place_ice_cream = ox.features.features_from_place(place_name, tags={"amenity": "ice_cream"})

# Projecting to Web-Mercator for more accurate centroids
place_ice_cream = place_ice_cream.to_crs("epsg:3857")

# Some of the ice cream shops return polygons instead of points, so we need to take their centroids
place_ice_cream["geometry"] = place_ice_cream.geometry.centroid

# Projecting back to lat/long
place_ice_cream = place_ice_cream.to_crs("epsg:4326")

# Display the result
print(place_ice_cream)

In [ ]:
place_ice_cream

In [ ]:
ice_cream_nodes = ox.distance.nearest_nodes(place_roads, place_ice_cream.geometry.x, place_ice_cream.geometry.y)
ice_cream_nodes

## Exercise
Plot the locations of the ice cream shops on the map of the roads

In [ ]:
#Import library
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Make sure ice cream shops and roads use the same CRS
needed_crs = place_roads_edges.crs
place_ice_cream_proj = place_ice_cream.to_crs(needed_crs)

# Create figure
fig, ax = plt.subplots(figsize=(11, 13))

# Plot NYC roads at the bottom layer
place_roads_edges.plot(
    ax=ax, 
    color="#006BB6", 
    linewidth=0.5, 
    alpha=0.6, 
    zorder=1,
    label = 'NYC Roads'
)

# Plot ice cream shop locations on the top layer
place_ice_cream_proj.plot(
    ax=ax, 
    color="#F58426", 
    markersize=15, 
    alpha=0.9, 
    zorder=2,
    label="Ice Cream Shops"
)

# Add legend
ax.legend(loc="upper left", frameon=True, facecolor="#BEC0C2")

# Add title
ax.set_title(
    "ice cream shops in da big apple", 
    fontsize=16, 
    fontweight="bold", 
    pad=15
)

# Remove axis for cleaner map
ax.set_axis_off()

# Show map
plt.show()

In [ ]:
ice_cream_nodes_copy = ice_cream_nodes
ice_cream_nodes = ice_cream_nodes[:30]

## Compute shortest path matrix

In [ ]:
shortest_path_matrix = np.zeros([len(ice_cream_nodes),len(ice_cream_nodes)])
for idx_i, orig in enumerate(ice_cream_nodes):
    shortest_paths = nx.single_source_dijkstra_path_length(place_roads, orig, weight='length')
    for idx_j, dest in enumerate(ice_cream_nodes):
        shortest_path_matrix[idx_i, idx_j] = shortest_paths[dest]
shortest_path_matrix

In [ ]:
'''
ice_cream_graph = nx.from_numpy_matrix(shortest_path_matrix, create_using=nx.MultiDiGraph)
'''
ice_cream_graph = nx.from_numpy_array(shortest_path_matrix, create_using=nx.MultiDiGraph)

In [ ]:
# new graph indexes from 0
ice_cream_graph.nodes

In [ ]:
# rename node labels using original labels
ice_cream_graph = nx.relabel_nodes(ice_cream_graph,{k:v for k, v in zip(ice_cream_graph.nodes, ice_cream_nodes)})
ice_cream_graph.nodes

In [ ]:
ice_cream_nodes=ice_cream_nodes_copy

## Exercise
Implement each of the following methods to see how good of a TSP path you can obtain.

## Method 1: Random
Let's start by setting a baseline; how well can we do by starting at a random node and choosing a random node out of the ones remaining each time? 

After you find the path, draw it on the map and print its length. (You don't need to draw the actual roads taken, just draw lines between the nodes.)

In [ ]:
# Select 30 ice cream shops
SAMPLE_SIZE = 30
SEED = 2
rng = np.random.default_rng(SEED)

all_ice_cream_nodes = list(ice_cream_nodes)

sample_idxs = rng.choice(
    len(all_ice_cream_nodes),
    size=SAMPLE_SIZE,
    replace=False
)
print("sample_idxs:", sample_idxs)

ice_cream_nodes_sample = [all_ice_cream_nodes[i] for i in sample_idxs]
place_ice_cream_sample = place_ice_cream.iloc[sample_idxs].copy()  

print(f"all shops: {len(ice_cream_nodes)}")
print(f"used for random: {len(ice_cream_nodes_sample)}")

In [ ]:
# Calculate shortest-path matrix for the 30 shops
n_stops = len(ice_cream_nodes_sample)

shortest_path_matrix = np.zeros((n_stops, n_stops))

for idx_i, orig in enumerate(ice_cream_nodes_sample):
    print(f"Calculating shortest paths from shop {idx_i + 1} of {n_stops}")
    shortest_paths = nx.single_source_dijkstra_path_length(place_roads, orig, weight='length')
    for idx_j, dest in enumerate(ice_cream_nodes_sample):
        shortest_path_matrix[idx_i, idx_j] = shortest_paths[dest]

In [ ]:
# Create a random TSP route
route_indices = list(range(n_stops))
rng.shuffle(route_indices)

# Complete the loop back to the origin
route_indices.append(route_indices[0])

# Map these back to our actual road-network node IDs
random_route_nodes = [ice_cream_nodes_sample[i] for i in route_indices]

In [ ]:
# Prepare route coordinates for plotting
route_x = [place_ice_cream_sample.geometry.iloc[i].x for i in route_indices]
route_y = [place_ice_cream_sample.geometry.iloc[i].y for i in route_indices]

In [ ]:
# Plot the random route
fig, ax = plt.subplots(figsize=(12, 12))

place_roads_edges.plot(
    ax=ax, 
    color="#006BB6", 
    linewidth=0.5, 
    alpha=0.5, 
    zorder=1,
    label="NYC Roads"
)

place_ice_cream_sample.plot(
    ax=ax,
    color="#F58426",
    markersize=35,
    zorder=3,
    label="Ice Cream Shops"
)

ax.plot(
    route_x, 
    route_y, 
    color="#FFDBBB", 
    linewidth=4, 
    alpha= 0.75, 
    zorder=2, 
    label="Random Route"
)

ax.legend(loc="upper left", frameon=True, facecolor="white")
ax.set_title("random route", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Print route length
total_length_meters = 0.0
for k in range(n_stops):
    idx_from = route_indices[k]
    idx_to = route_indices[k + 1]
    total_length_meters += shortest_path_matrix[idx_from, idx_to]

print(f"length:{total_length_meters:.2f} meters")

## Method 2: Greedy
Now, let's try to choose nodes more intelligently: start at a random node again, but instead of choosing a random node each time, always choose the node closest to the current node each time.

Again, draw the path on the map and print its length.

In [ ]:
# Select 30 ice cream shops
SAMPLE_SIZE = 30
SEED = 2
rng = np.random.default_rng(SEED)

all_ice_cream_nodes = list(ice_cream_nodes)

sample_idxs = rng.choice(
    len(all_ice_cream_nodes),
    size=SAMPLE_SIZE,
    replace=False
)
print("sample_idxs:", sample_idxs)

ice_cream_nodes_sample = [all_ice_cream_nodes[i] for i in sample_idxs]
place_ice_cream_sample = place_ice_cream.iloc[sample_idxs].copy()  

print(f"all shops: {len(ice_cream_nodes)}")
print(f"used for random: {len(ice_cream_nodes_sample)}")

In [ ]:
# Calculate shortest-path matrix (or use the one you developed earlier)
n_stops = len(ice_cream_nodes_sample)

shortest_path_matrix = np.zeros((n_stops, n_stops))

for idx_i, orig in enumerate(ice_cream_nodes_sample):
    print(f"Calculating shortest paths from shop {idx_i + 1} of {n_stops}")
    shortest_paths = nx.single_source_dijkstra_path_length(place_roads, orig, weight='length')
    for idx_j, dest in enumerate(ice_cream_nodes_sample):
        shortest_path_matrix[idx_i, idx_j] = shortest_paths[dest]

In [ ]:
# Create greedy TSP route
start_idx = rng.choice(n_stops)

unvisited = list(range(n_stops))
unvisited.remove(start_idx)

route_indices = [start_idx]

current_idx = start_idx
while len(unvisited) > 0:
  
    next_idx = min(unvisited, key=lambda x: shortest_path_matrix[current_idx, x])
    
    
    route_indices.append(next_idx)
    unvisited.remove(next_idx)
    current_idx = next_idx


route_indices.append(start_idx)

greedy_route_nodes = [ice_cream_nodes_sample[i] for i in route_indices]

In [ ]:
# Prepare route coordinates for plotting
route_x = [place_ice_cream_sample.geometry.iloc[i].x for i in route_indices]
route_y = [place_ice_cream_sample.geometry.iloc[i].y for i in route_indices]

In [ ]:
# Plot the greedy route
fig, ax = plt.subplots(figsize=(12, 12))

place_roads_edges.plot(
    ax=ax, 
    color="#006BB6", 
    linewidth=0.5, 
    alpha=0.5, 
    zorder=1,
    label="NYC Roads"
)

place_ice_cream_sample.plot(
    ax=ax,
    color="#F58426",
    markersize=35,
    zorder=3,
    label="Ice Cream Shops"
)

ax.plot(
    route_x, 
    route_y, 
    color="#FFDBBB", 
    linewidth=4, 
    alpha= 0.75, 
    zorder=2, 
    label="Random Route"
)

ax.legend(loc="upper left", frameon=True, facecolor="white")
ax.set_title("greedy route", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Print route length
total_length_meters = 0.0
for k in range(n_stops):
    idx_from = route_indices[k]
    idx_to = route_indices[k + 1]
    total_length_meters += shortest_path_matrix[idx_from, idx_to]

print(f"length:{total_length_meters:.2f} meters")

## Method 3: Random with 2-opt swapping (Optional Challenging)

You may have noticed that both paths contain a lot of edges that cross each other, which is nonideal. However, there exists an algorithm to remove all the paths that cross each other from a Hamiltonian cycle: the [2-opt](https://en.wikipedia.org/wiki/2-opt) algorithm. We can use that to our advantage here.

Start by generating a random Hamiltonian cycle like in method 1, but this time, use the 2-opt algorithm to optimize it further. Again, draw it on the map and print its length.

In [ ]:
# Select 30 ice cream shops — same as Method 1 and Method 2


In [ ]:
# Calculate shortest-path matrix — same as Method 1 and Method 2


In [ ]:
# Create initial random route — similar to Method 1


In [ ]:
# Improve the random route using 2-opt swapping


In [ ]:
# Prepare route coordinates for plotting — similar to Method 1 and Method 2


In [ ]:
# Plot the 2-opt route — similar to Method 1 and Method 2


In [ ]:
# Print route length — similar to Method 1 and Method 2


## Method 4: Open-ended( Optional Challenging)

Although the 2-opt swaps reduce the length of the Hamiltonian cycle by quite a lot, they almost never provide the optimal solution. See if you can use another method to produce a Hamiltonian cycle shorter than the one you got with method 3. Some options to explore include:

- [3-opt](https://en.wikipedia.org/wiki/3-opt)
- [Multi-fragment algorithm](https://en.wikipedia.org/wiki/Multi-fragment_algorithm) with 2- or 3-opt swapping
- [Simulated annealing](https://en.wikipedia.org/wiki/Simulated_annealing)

The [TSP Wikipedia page](https://en.wikipedia.org/wiki/Travelling_salesman_problem) has many other algorithms that could be of use to you as well.


In [ ]:
# Select 30 ice cream shops — same as Method 1, Method 2, and Method 3



In [ ]:
# Calculate the shortest road-network distance between every pair of selected ice cream shops



In [ ]:
# create a a random TSP route



In [ ]:
# Create a 2-opt route first so Method 4 can compare against Method 3



In [ ]:
# Improve the 2-opt route using 3-opt swapping




In [ ]:
# Convert route indexes back to original road-network node IDs


In [ ]:
#plot


In [ ]:
# calculate route length
